# Lab 1: Running and Querying Open-Source LLMs Locally

**Time:** about 60 minutes

This morning you saw how transformers work: tokens, embeddings, attention, layer after layer. This afternoon you'll
run some, on your own machine, two ways:

- through **Ollama**, which is how you'd serve a model to other applications inside the Fund,
- through **Hugging Face Transformers**, which is how you'd look inside a model or fine-tune it (Day 8).

Along the way you'll measure the things that decide whether an LLM (large language model) is usable in practice: speed,
tokens, how settings change the answers, and how confidently a model can be wrong.

### What is an LLM, in one paragraph?

A **large language model** is a transformer (Day 6) trained on a huge amount of text to do one thing: given some text,
predict the next token. Do that once and you get one word-piece. Do it in a loop, adding each prediction to the text,
and you get sentences, answers, summaries and code. Everything an LLM does, however clever it looks, is built from
that one step repeated.

**Open-source** (or "open-weight") models are ones whose trained weights you can download and run yourself, on your own
computer, so no text ever leaves the Fund. That's the opposite of cloud services like ChatGPT, where your text is sent
to someone else's servers.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **LLM** | large language model: a transformer trained to predict the next token, used to generate text |
| **open-weight model** | a model you can download and run on your own machine |
| **Ollama** | a free program that downloads open models and runs them, with a simple web interface for other programs |
| **API** | application programming interface: a way for one program to send requests to another |
| **JSON** | a plain-text format for structured data: `{"name": "value"}` |
| **prompt** | the text you send to the model |
| **role** | who a chat message is from: `system` (instructions), `user` (you) or `assistant` (the model) |
| **token** | a piece of a word; models read, write and are priced in tokens |
| **context window** | the most tokens a model can consider at once (prompt plus answer) |
| **parameters** | the model's learned numbers; 1.5B = 1.5 billion |
| **quantisation** | storing each parameter with fewer bits (e.g. 4 instead of 16), so the model is smaller and faster |
| **temperature** | how much randomness is used when choosing the next token; 0 = none |
| **hallucination** | a fluent, confident answer that is wrong or made up |
| **chat template** | the special markers a model expects around each message |
| **logits** | the raw score the model gives every possible next token |

## Where this lab fits

**What you've done so far.** For six days you built models that each do **one** job: score a claim, forecast a series,
detect an unusual payment, route a complaint, extract a member number. On Day 6 you met transformers: tokens, attention,
pre-training and fine-tuning, in a model with about 135 million parameters.

**What you'll do now.** You'll run large language models, transformers with billions of parameters that do many jobs
from a written instruction, entirely on your own machine. You'll call them through Ollama's local web interface, measure
tokens and speed, see how temperature changes answers, watch a model answer policy questions confidently and wrongly,
then open a small model with Hugging Face to see the next-token prediction underneath.

**Why it matters.** The Fund holds personal and financial data about millions of members. Before anyone uses an LLM on
that data, the Fund needs to know which models it can run under its own control, how fast they are on its own hardware,
and what they get wrong. That's the foundation for every LLM decision in Days 7-10.

**What happens next.** Lab 2 learns to write prompts that make these models useful and safe for Fund tasks. Lab 3 scores
several models on the Fund's own tasks to choose a shortlist.

### Theory: why run models locally?

There are two ways to use an LLM:

| | Cloud service (e.g. ChatGPT, hosted APIs) | Local / self-hosted (today) |
|---|---|---|
| Where the text goes | to another company's servers | stays on the Fund's machines |
| Model size | the largest, strongest models | what your hardware can hold |
| Cost | pay per token, every time | hardware and electricity, once set up |
| Control | provider can change or retire the model | you choose when to update |
| Setup | an account and a key | installing and running the model yourself |

For member data, the first row usually decides it. Sending a member's message, claim or national ID to an outside
service is a data-protection question (Day 6's redaction lab raised it), and for many uses the answer is simply "not
without strong safeguards". Local models avoid that question. The price is that you must work with smaller models, which
is why measuring what they can and can't do matters so much.

## 0. Set up today's environment (run this cell first)

You don't need to create a folder or an environment by hand today. The cell below does it for you:

1. It finds the environment you created on Day 1 (`DAY1/Labs/work/fund-ai/.venv`). If it isn't there, it creates it.
2. It installs today's libraries into it from ``requirements.txt``. The first time takes a few minutes; after that, seconds.
3. If this machine has an NVIDIA GPU (graphics processing unit), it switches PyTorch to its GPU build, so training runs much faster.
4. It connects the environment to Jupyter as **Python (fund-ai)**, and tells you if this notebook isn't using it yet.

It's safe to run again at any time. If it prints instructions to switch the kernel, follow them and run it once more.

You don't need to read the code in this cell. It's housekeeping, not part of today's lesson.

In [2]:
# ---- Setup for Day 7. Safe to run more than once. -------------------------------------------------------------
COURSE_ENV = None        # None = use the Day 1 environment. Or set a folder, e.g. Path.home() / "fund-ai-venv"
REQUIREMENTS = "requirements.txt"
EXTRAS = ['ollama:qwen2.5:1.5b,llama3.2:1b,gemma2:2b']

import os, shutil, subprocess, sys
from pathlib import Path

labs = Path.cwd()
root = next((p for p in [labs, *labs.parents] if (p / "DAY1").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the course folder (the one that contains DAY1, DAY2, ...).")
env = Path(COURSE_ENV) if COURSE_ENV else root / "DAY1" / "Labs" / "work" / "fund-ai" / ".venv"
py = env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def run(*cmd, quiet=False):
    """Run a command; show its output only if it fails."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0 and not quiet:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if "path" in (r.stderr + r.stdout).lower() and os.name == "nt":
            print("\nIf the error mentions a path that is too long: Windows 'long paths' are switched off on this machine.\n"
                  "Ask ICT to enable them, or set COURSE_ENV above to a short folder, e.g. Path.home() / 'fund-ai-venv'.")
        raise RuntimeError("A setup step failed. The details are printed above.")
    return r


# 1. The course environment
if py.exists():
    print(f"- course environment found: {env}")
else:
    print(f"- creating the course environment (first time only): {env}")
    run(getattr(sys, "_base_executable", sys.executable), "-m", "venv", env)
    run(py, "-m", "pip", "install", "-q", "--upgrade", "pip")

# 2. Today's libraries
req = labs / REQUIREMENTS
print(f"- installing today's libraries from {REQUIREMENTS} ...")
run(py, "-m", "pip", "install", "-q", "-r", req)

# 3. Use the GPU if this machine has an NVIDIA one
if "torch" in req.read_text() and shutil.which("nvidia-smi"):
    if run(py, "-c", "import torch; print(torch.cuda.is_available())").stdout.strip() != "True":
        version = run(py, "-c", "import torch; print(torch.__version__.split('+')[0])").stdout.strip()
        print(f"- NVIDIA GPU found: switching PyTorch {version} to its GPU build (about 2.5 GB, first time only) ...")
        run(py, "-m", "pip", "uninstall", "-y", "-q", "torch")
        run(py, "-m", "pip", "install", "-q", f"torch=={version}", "--index-url", "https://download.pytorch.org/whl/cu124",
            "--extra-index-url", "https://pypi.org/simple")
    print("- PyTorch will use the GPU")

# 4. Anything else today needs
for extra in EXTRAS:
    kind, name = extra.split(":", 1)
    if kind == "spacy" and run(py, "-c", f"import {name}", quiet=True).returncode != 0:
        print(f"- downloading the spaCy model {name} ...")
        run(py, "-m", "spacy", "download", name)
    if kind == "ollama":
        import json, urllib.request
        try:
            tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3))["models"]
            have = {m["name"] for m in tags}
            wanted = [m for m in name.split(",") if m]
            missing = [m for m in wanted if m not in have]
            print(f"- Ollama is running with {len(have)} model(s)" + (f"; still to pull: {', '.join(missing)}" if missing else ""))
            for m in missing:
                print(f"    in a terminal: ollama pull {m}")
        except OSError:
            print("- Ollama isn't running. Start the Ollama app (or run `ollama serve` in a terminal) before the LLM parts.")

# 5. Connect the environment to Jupyter
run(py, "-m", "ipykernel", "install", "--user", "--name", "fund-ai", "--display-name", "Python (fund-ai)")

# 6. Is this notebook using it?
if Path(sys.prefix).resolve() == env.resolve():
    print("\nReady. This notebook is running in the course environment. Carry on with the next cell.")
else:
    print("\nAlmost there. This notebook is running a different Python:")
    print(f"    {sys.executable}")
    print("Switch it to the course environment, then run this cell once more:")
    print("    1. Click the kernel name in the top right of the notebook, then 'Select Another Kernel'.")
    print("    2. Choose 'Jupyter Kernel' -> 'Python (fund-ai)'.")
    print("       (Or press Ctrl+Shift+P, run 'Python: Select Interpreter', choose 'Enter interpreter path...'")
    print("        and paste the path below, the same way you did on Day 1.)")
    print(f"    {py}")

- course environment found: /Users/daud.mabena/Documents/DUBAI/Training1910Labs/DAY1/Labs/work/fund-ai/.venv
- installing today's libraries from requirements.txt ...
- Ollama is running with 6 model(s)

Ready. This notebook is running in the course environment. Carry on with the next cell.


## Imports

Only two new things:

- `requests`: sends web requests. Ollama runs a small web server on your machine, and Python talks to it the same way a
  browser talks to a website: by sending a request to an address and reading the reply.
- `OLLAMA_URL`: that address. `localhost` means "this computer", and `11434` is the **port** (a numbered door) Ollama
  listens on. `os.environ.get` lets someone override it with an environment variable, for example to use an Ollama
  server elsewhere on the Fund's network.

In [3]:
import os
import time
import warnings

import pandas as pd
import requests              # sends web requests: this is how Python talks to Ollama

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 170)
warnings.filterwarnings("ignore")        # hide library notices

# Ollama runs a small web server on your machine, on port 11434.
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")

Check that Ollama is running by asking for its version. `requests.get(address)` sends a request and waits for the reply;
`.json()` turns the reply's JSON text into a Python dictionary. If Ollama isn't running, the request fails with a
`ConnectionError`, and the `except` part prints what to do instead of crashing.

In [4]:
# Is Ollama running? Ask it for its version.
try:
    response = requests.get(OLLAMA_URL + "/api/version", timeout=5)      # give up after 5 seconds
    print("Ollama:", response.json())
except requests.exceptions.ConnectionError:
    print("Can't reach Ollama. Start the Ollama app (or run `ollama serve` in a terminal), then rerun this cell.")

Ollama: {'version': '0.32.0'}


> **Where we are.** Ollama is running and answering on `localhost:11434`. Next, see which models are on your machine.

## 1. Which models do you have?

Ollama keeps a list of the models you've downloaded (with `ollama pull <name>` in a terminal). The address `/api/tags`
returns that list. Each model comes with details; the loop picks out the useful ones into a table, one row per model.
`.get(...)` reads a value from a dictionary, returning `None` instead of an error if it's missing.

In [5]:
# /api/tags lists the models you've pulled. The reply is JSON, which .json() turns into Python dictionaries and lists.
response = requests.get(OLLAMA_URL + "/api/tags")
installed = response.json()["models"]

# One row per model.
rows = []
for m in installed:
    rows.append({
        "model": m["name"],
        "size_gb": round(m["size"] / 1e9, 2),                       # bytes -> gigabytes
        "parameters": m["details"].get("parameter_size"),
        "quantisation": m["details"].get("quantization_level"),
        "family": m["details"].get("family"),
    })

models = pd.DataFrame(rows).sort_values("size_gb")
models

,model,size_gb,parameters,quantisation,family
3,qwen3-embedding:0.6b,0.64,595.78M,Q8_0,qwen3
2,qwen2.5:1.5b,0.99,1.5B,Q4_K_M,qwen2
1,llama3.2:1b,1.32,1.2B,Q8_0,llama
0,gemma2:2b,1.63,2.6B,Q4_0,gemma2
4,qwen3:4b,2.50,4.0B,Q4_K_M,qwen3
5,qwen2.5-coder:7b,4.68,7.6B,Q4_K_M,qwen2


Two columns deserve a word:

- **parameters** (e.g. 1.5B = 1.5 billion). Roughly, more parameters means more knowledge and better reasoning, and
  also more memory and slower answers.
- **quantisation** (e.g. Q4_K_M). The weights are stored with 4 bits instead of 16, which makes the model about four
  times smaller with a modest loss of quality. That's why a 3-billion-parameter model fits in about 2 GB and runs on a
  laptop.

A quick sum shows why quantisation matters: 1.5 billion parameters × 16 bits = 24 billion bits = 3 GB. At 4 bits it's
about 0.75 GB, plus some overhead. Memory, not processing power, is usually what limits which models a machine can run.

Pick the model you'll use for most of this lab. The smallest one answers fastest. Change it if you pulled bigger ones.

In [6]:
if "qwen2.5:1.5b" in list(models["model"]):
    MODEL = "qwen2.5:1.5b"
else:
    MODEL = models["model"].iloc[0]          # otherwise the smallest model you have
print("Using", MODEL)

Using qwen2.5:1.5b


> **Where we are.** You've listed your models, seen how parameters and quantisation decide their size, and chosen
> `MODEL` for the rest of the lab. Now talk to it.

### Theory: how a program talks to a model server

Ollama runs as a small **server**: a program that waits for requests and answers them. Your notebook is the **client**.
They talk over **HTTP**, the same way a browser talks to a website:

```
notebook (client)  ──  POST /api/chat  {model, messages, options}  ──►  Ollama (server)
                   ◄──  200 OK  {message, token counts, timings}   ──
```

- The **address** says which server (`localhost:11434`) and which service (`/api/chat`).
- The **body** is JSON: a plain-text dictionary of what you want.
- The **reply** is JSON too, with the answer and measurements.

Because it's ordinary HTTP, any program in any language can use the same model: a Python notebook today, a Java claims
system or a web portal tomorrow. That's what makes a model server useful inside an organisation, and it's the pattern
you'll build yourself on Day 9.

## 2. Your first request

Ollama's chat API (application programming interface: a way for one program to call another) takes a list of
messages, each with a **role**:

- `system`: standing instructions: who the assistant is, how it should behave;
- `user`: the question or task;
- `assistant`: the model's earlier replies, if you're continuing a conversation.

Build one request by hand. It's a dictionary that `requests` sends as JSON. The `options` control how the answer is
generated:

- `temperature`: how much randomness to use (Section 4);
- `num_predict`: the most tokens the answer may have, so a model can't ramble forever;
- `seed`: fixes any randomness, so results can be repeated.

`requests.post` sends the request to the `/api/chat` address. (A **POST** request carries data, here your messages; a
**GET**, as above, only asks for something.) `timeout=300` waits up to 5 minutes, because the first answer can be slow.

In [7]:
request_body = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "In two sentences, what is a social security fund?"},
    ],
    "stream": False,                 # wait for the whole answer instead of receiving it word by word
    "options": {
        "temperature": 0.0,          # no randomness (more on this in Section 4)
        "num_predict": 400,          # at most 400 tokens in the answer
        "seed": 42,
    },
}

start = time.perf_counter()                       # start the stopwatch
response = requests.post(OLLAMA_URL + "/api/chat", json=request_body, timeout=300)
seconds = time.perf_counter() - start

r = response.json()
print(r["message"]["content"])                    # the answer text
print(f"\n({seconds:.1f} seconds)")

A social security fund is an account or pool of money set aside for the benefit of individuals who have paid into it through contributions or taxes. It's used to provide financial support and protection in case of retirement, disability, unemployment, or death.

(3.4 seconds)


The reply carries more than the text. Look at everything that came back:

In [8]:
list(r.keys())

['model',
 'created_at',
 'message',
 'done',
 'done_reason',
 'total_duration',
 'load_duration',
 'prompt_eval_count',
 'prompt_eval_duration',
 'eval_count',
 'eval_duration']

You'll send many requests, so put those steps in a function. It builds the message list (with an optional system
message), sends the request, and returns the whole reply, plus the time it took. Every setting has a sensible default,
so `chat("question")` is all you need for a simple call.

`fmt` is used in Lab 2, to force the model to reply in a fixed JSON structure.

In [9]:
def chat(prompt, system=None, model=MODEL, temperature=0.0, max_tokens=400, fmt=None, seed=42):
    """Send one prompt to a model in Ollama and return its full reply (a dictionary)."""
    # The messages: an optional system instruction, then the user's prompt.
    messages = []
    if system is not None:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    body = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": max_tokens, "seed": seed},
    }
    if fmt is not None:
        body["format"] = fmt         # optional: force the reply into a JSON structure (Lab 2)

    # Send it, and time how long the reply takes.
    start = time.perf_counter()
    reply = requests.post(OLLAMA_URL + "/api/chat", json=body, timeout=300).json()
    reply["wall_seconds"] = time.perf_counter() - start
    return reply


r = chat("In two sentences, what is a social security fund?")
print(r["message"]["content"])

A social security fund is an account or pool of money set aside for the benefit of individuals who have paid into it through contributions or taxes. It's used to provide financial support and protection in case of retirement, disability, unemployment, or death.


The response also carries measurements. These are what you'd monitor in production:

- **input tokens:** how long your prompt was, in tokens;
- **output tokens:** how long the answer was;
- **tokens per second:** generation speed. Roughly, people read at 5 to 10 tokens a second, so anything faster feels
  instant in a chat;
- **total seconds:** how long the user waited.

In [10]:
input_tokens = r["prompt_eval_count"]          # tokens in your prompt (plus the chat template)
output_tokens = r["eval_count"]                # tokens in the answer
generation_seconds = r["eval_duration"] / 1e9  # Ollama reports nanoseconds; 1e9 of them = 1 second

print("input tokens:     ", input_tokens)
print("output tokens:    ", output_tokens)
print("tokens per second:", round(output_tokens / generation_seconds, 1))
print("total seconds:    ", round(r["wall_seconds"], 2))

input tokens:      40
output tokens:     50
tokens per second: 109.8
total seconds:     0.82


The first call is slower: Ollama loads the model into memory, then keeps it there for a few minutes. Run the two cells
above again and compare.

> **Where we are.** You've sent a request by hand, wrapped it in a `chat` function, and seen the reply's measurements:
> tokens in, tokens out, speed and waiting time. Tokens come up in every limit and every price, so look at them more
> closely.

## 3. Tokens, and why Swahili costs more

An LLM reads and writes tokens, not words. Every limit and every price is in tokens: the **context window** (how much
text the model can consider at once), the speed (tokens per second), and the bill if you ever use a cloud model
(price per million tokens). So it's worth knowing how many tokens your text becomes.

A neat trick to count them: send each sentence with the instruction "Reply with the single word OK" and
`max_tokens=3`. The answer is tiny, and the reply's `prompt_eval_count` tells you how many tokens the prompt was.
`len(text.split())` counts the words for comparison.

In [11]:
# The same two sentences in English and in Swahili.
sentences = [
    ("English", "My employer deducts contributions every month but does not send them to the Fund."),
    ("Swahili", "Mwajiri wangu anakata michango kila mwezi lakini hapeleki kwenye Mfuko."),
    ("English", "I submitted my maternity claim three weeks ago and I have not been paid."),
    ("Swahili", "Niliwasilisha madai ya fao la uzazi wiki tatu zilizopita na sijalipwa."),
]

rows = []
for language, text in sentences:
    # Ask for a one-word answer: we only want to know how many tokens the prompt was.
    r = chat("Reply with the single word OK.\n\n" + text, max_tokens=3)
    rows.append({"language": language, "words": len(text.split()), "prompt tokens": r["prompt_eval_count"]})

pd.DataFrame(rows)

,language,words,prompt tokens
0,English,14,52
1,Swahili,10,64
2,English,14,51
3,Swahili,11,64


The prompt token count includes a small fixed overhead (the chat template and the instruction), but compare the pairs:
the same meaning costs more tokens in Swahili. Most models' vocabularies were built from text that's overwhelmingly
English, so Swahili words are split into more pieces.

In practice that means Swahili conversations fill the context window sooner, run a little slower, and cost more on a
per-token cloud service. It's also a hint that a model's Swahili *quality* may lag its English. You'll test that in
Lab 3.

> **Where we are.** You've seen that the same meaning takes more tokens in Swahili, with consequences for length limits,
> speed and cost. Next, the settings that change *what* the model says.

## 4. Settings that change the answer

**Temperature** controls randomness. Remember that at every step the model gives a probability to every possible next
token (you'll see this in Section 6). Temperature decides how that probability is used:

- at **0**, the model always picks its single most likely next token, so the same prompt gives the same answer;
- at higher values (0.7, 1.0), less likely tokens get chosen more often: more varied wording, and less predictable.

Ask for the same SMS twice at temperature 0:

In [14]:
prompt = "Write a one-sentence SMS reminding an employer that March contributions are due by 30 April."

# Twice at temperature 0: expect the same answer both times.
for attempt in [1, 2, 3, 4]:
    answer = chat(prompt, temperature=0.0)["message"]["content"].strip()      # strip() removes spaces at the ends
    print("temperature 0.0:", answer, "\n")

temperature 0.0: "Reminder: Your March contributions are due by 30th April; don't miss this important deadline!" 

temperature 0.0: "Reminder: Your March contributions are due by 30th April; don't miss this important deadline!" 

temperature 0.0: "Reminder: Your March contributions are due by 30th April; don't miss this important deadline!" 

temperature 0.0: "Reminder: Your March contributions are due by 30th April; don't miss this important deadline!" 



Now three times at temperature 1. Each call uses a different `seed`, so the randomness differs each time.

In [18]:
# Three times at temperature 1, each with a different random seed: expect three different answers.
for seed in [1, 2, 3]:
    answer = chat(prompt, temperature=0.0, seed=seed)["message"]["content"].strip()
    print("temperature 1.0:", answer, "\n")

temperature 1.0: "Reminder: Your March contributions are due by 30th April; don't miss this important deadline!" 

temperature 1.0: "Reminder: Your March contributions are due by 30th April; don't miss this important deadline!" 

temperature 1.0: "Reminder: Your March contributions are due by 30th April; don't miss this important deadline!" 



For the Fund's work (answering policy questions, extracting fields, classifying messages) you want **temperature 0**:
the same input should give the same output, so results can be tested, audited and explained. Higher temperatures are for
drafting varied text a person will edit anyway.

**Your turn (1).** `max_tokens` (Ollama calls it `num_predict`) caps the length of the answer. Ask the model to *"List
all the benefits a social security fund might pay"* with `max_tokens=30` and then 300. What happens at 30? Why does it
matter for an application that expects complete JSON (JavaScript Object Notation, a plain-text format for structured
data)?

In [22]:
# Your code here
prompt_o = "List all the benefits a social security fund might pay"
for attempt in [10, 30, 300]:
    answer = chat(prompt_o, temperature=0.0, max_tokens=attempt)["message"]["content"].strip()      # strip() removes spaces at the ends
    print("T:", attempt, "A:", answer, "\n")

T: 10 A: A social security fund typically provides several key benefits to 

T: 30 A: A social security fund typically provides several key benefits to individuals and families. Here is a list of some common benefits:

1. **Healthcare Coverage**: 

T: 300 A: A social security fund typically provides several key benefits to individuals and families. Here is a list of some common benefits:

1. **Healthcare Coverage**: Many social security funds offer comprehensive healthcare coverage, including hospitalization, outpatient services, and prescription drugs.

2. **Pension Benefits**: Social security funds provide retirement income through pensions or annuities, which are designed to help individuals maintain their standard of living after they stop working.

3. **Disability Insurance**: Some funds offer disability insurance that provides financial support if an individual becomes unable to work due to illness or injury.

4. **Survivor Benefits**: These benefits cover the loss of income for 

> **Where we are.** You know to use temperature 0 for the Fund's tasks, and to set the length limit with room to spare.
> The model's answers are now repeatable. Whether they're *right* is a different question.

### Theory: why LLMs make things up

An LLM was trained to produce **likely text**, not **true text**. During training it saw countless pages that state
penalty rates, deadlines and percentages, and it learned what such statements look like. Asked about NSSF Tanzania's
penalty, it produces a statement that looks right: fluent, specific and confident, built from whatever patterns it
learned. Nothing inside the model checks the claim against a source.

Three things make it worse for the Fund:

- **Little training text about the Fund.** Specific Tanzanian rules are rare on the internet, so the model has little
  to go on and fills the gap with patterns from other countries' schemes.
- **Rules change.** Even a model that once saw the right rate won't know about a circular issued after its training.
- **Small models know less.** The models you can run on a laptop have far less room to store facts than the largest
  cloud models.

The fix isn't to find a model that never makes things up. It's to give the model the right text at question time and
instruct it to answer only from that, which is Lab 2's grounding and Day 9's retrieval.

## 5. The problem you must design around: confident wrong answers

Ask a policy question with no reference material. The model has to answer from whatever it absorbed during training,
which almost certainly included little about NSSF Tanzania's specific rules. Watch what it does:

In [23]:
questions = [
    "What is the penalty for late payment of NSSF contributions in Tanzania?",
    "How many months must a member contribute to NSSF Tanzania to qualify for a pension?",
    "Does NSSF Tanzania pay compensation for injuries at work?",
]

for question in questions:
    answer = chat(question, max_tokens=150)["message"]["content"].strip()
    print("Q:", question)
    print("A:", answer)
    print()

Q: What is the penalty for late payment of NSSF contributions in Tanzania?
A: As an AI language model, I cannot provide information about specific countries or regions. Please note that different countries have different laws and regulations regarding social security contributions. It's best to consult with a legal professional or relevant government agency for accurate and up-to-date information on penalties for late payment of NSSF contributions in Tanzania.

Q: How many months must a member contribute to NSSF Tanzania to qualify for a pension?
A: To determine how long a member needs to contribute to the National Social Security Fund (NSSF) in Tanzania to qualify for a pension, we need to look at the specific rules and regulations of the fund. 

In general, most pension schemes require a certain number of years of contributions before an individual can receive benefits. For NSSF Tanzania, the minimum contribution period required is typically 15 years.

However, it's important to chec

Read the answers against what you know about the Fund, and against the training handbook in `handbook/`. With a small
model answering from memory you'll typically see a mixture of four things:

- **a specific, fluent, wrong answer**: an invented percentage, or rules borrowed from another country's scheme;
- **an answer that happens to be right**, which is no safer, because nothing tells you *which* answers are the lucky ones;
- **a vague hedge** ("penalties can vary, check the official website"), which is honest but not useful;
- **a refusal or a strange misreading** of a simple question.

Try the same questions with a different model, or rerun them tomorrow after a model update, and the mixture changes.
The confident wrong answers are often called **hallucination**. It isn't a bug that a bigger model fixes completely.
It's how the technology works: the model produces likely-sounding text, not checked facts. For the Fund that means one
firm design rule:

> An LLM must never be the source of truth for policy. It should answer **from documents you give it**, and say so when
> the documents don't contain the answer.

That's what Lab 2 practises and what Day 9 (retrieval-augmented generation) builds properly.

> **Where we are.** You've seen the problem every LLM application must design around: confident wrong answers on policy.
> So far the models have been a black box behind Ollama. Now open one up.

## 6. The same kind of model through Hugging Face Transformers

Ollama is convenient for serving. Hugging Face Transformers gives you the model itself, so you can look inside it and, on
Day 8, fine-tune it. Load a small instruction-tuned model (Qwen2.5-0.5B-Instruct, about 1 GB). It uses the GPU if
there is one.

**Instruction-tuned** means the model has had extra training on examples of instructions and good answers, so it
behaves like an assistant. A "base" model, without that step, just continues whatever text you give it.

Two classes do the loading, as on Day 6: `AutoTokenizer` for the tokenizer, and `AutoModelForCausalLM` for the model.
**Causal LM** (causal language model) means a model that predicts the next token from the ones before it, which is
what every chat model is. On a GPU the weights are loaded as 16-bit numbers (`float16`), which halves the memory.

In [24]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# GPU if there is one. On a GPU, 16-bit numbers (float16) halve the memory.
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)                                     # downloads the first time
hf_model = AutoModelForCausalLM.from_pretrained(HF_MODEL, torch_dtype=dtype).to(device)
print(HF_MODEL, "on", device)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Qwen/Qwen2.5-0.5B-Instruct on cpu


How big is it, and what shape? Every PyTorch model can list its parameters (the weight tensors); `numel()` counts the
numbers in each, and adding them up gives the total. The model's `config` describes its shape.

In [25]:
# How big is it, and what shape?
n_parameters = 0
for parameter in hf_model.parameters():
    n_parameters = n_parameters + parameter.numel()      # numbers in this weight tensor

cfg = hf_model.config
print("parameters:              ", round(n_parameters / 1e6), "million")
print("layers:                  ", cfg.num_hidden_layers)
print("attention heads per layer:", cfg.num_attention_heads)
print("hidden size:             ", cfg.hidden_size)
print("vocabulary:              ", cfg.vocab_size, "tokens")
print("context window:          ", cfg.max_position_embeddings, "tokens")

parameters:               494 million
layers:                   24
attention heads per layer: 14
hidden size:              896
vocabulary:               151936 tokens
context window:           32768 tokens


Those numbers map straight onto this morning's diagram. Every token becomes a vector of `hidden size` numbers (the
embedding). That vector passes through `layers` transformer blocks. In each block, `attention heads` let every token look
at every other token to decide which ones matter for its meaning. The final layer turns the vector back into a score for
each of the `vocabulary` tokens, and the most likely one comes out.

Generation is that step in a loop: predict one token, append it, predict the next.

First, the **chat template**: the special markers the model was trained to see around each role. `apply_chat_template`
wraps your messages in them. `tokenize=False` returns the text so you can read it; `add_generation_prompt=True` adds the
marker that says "now the assistant speaks".

In [26]:
messages = [{"role": "user", "content": "Hello"}]
template_text = hf_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(template_text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant



Ollama adds these markers for you. With Transformers you apply them yourself, and getting them wrong is a common reason
a model gives strange answers. Now generate an answer, step by step:

1. wrap the question in the chat template;
2. turn the text into token ids (as PyTorch tensors, `return_tensors="pt"`) on the same device as the model;
3. `generate` runs the predict-one-token loop, up to 120 new tokens. `do_sample=False` always takes the most likely
   token, the same as temperature 0;
4. the output contains the prompt's tokens too, so keep only the new ones and `decode` them back into text.

In [ ]:
# 1. Wrap the question in the chat template.
messages = [{"role": "user", "content": "In one sentence, what does a pension fund do?"}]
text = hf_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 2. Turn the text into token ids, as PyTorch tensors, on the same device as the model.
inputs = hf_tok(text, return_tensors="pt").to(device)
n_prompt_tokens = inputs["input_ids"].shape[1]

# 3. Generate up to 120 new tokens. do_sample=False = always the most likely token (like temperature 0).
with torch.no_grad():
    output_ids = hf_model.generate(**inputs, max_new_tokens=120, do_sample=False)

# 4. The output contains the prompt too. Keep only the new tokens, and turn them back into text.
new_ids = output_ids[0, n_prompt_tokens:]
print(hf_tok.decode(new_ids, skip_special_tokens=True))

A pension fund is an investment vehicle that pools the


### Watching the next-token prediction

Look at what the model thinks could come next after a short prompt, and how likely each option is. Instead of
`generate`, run the model once (`hf_model(**inputs)`): it returns **logits**, a score for every vocabulary token at every
position in the text.

- `logits[0, -1]` takes the first (only) message and its **last** position: the scores for what comes next;
- softmax turns the scores into probabilities that add up to 1 (as on Day 6);
- `torch.topk(..., 8)` finds the 8 highest probabilities and their token ids;
- `decode` turns each id back into text; `repr` shows it in quotes, so you can see leading spaces.

In [29]:
messages = [{
    "role": "user", 
    "content": "Complete: The Fund pays pensions every"
}]
text = hf_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = hf_tok(text, return_tensors="pt").to(device)

# One pass through the model. logits = a score for every token in the vocabulary, at every position.
with torch.no_grad():
    logits = hf_model(**inputs).logits

# We want the scores at the LAST position (what comes next), turned into probabilities.
last_scores = logits[0, -1].float()
probabilities = torch.softmax(last_scores, dim=-1)

# The 8 most likely next tokens.
top = torch.topk(probabilities, 8)
rows = []
for token_id, probability in zip(top.indices, top.values):
    rows.append({"next token": repr(hf_tok.decode([int(token_id)])), "probability": round(float(probability), 3)})
pd.DataFrame(rows)

,next token,probability
0,'The',0.400
1,'year',0.102
2,'the',0.044
3,'Every',0.041
4,'every',0.020
5,'to',0.020
6,'th',0.018
7,'To',0.017


That table is the whole trick behind an LLM: a probability for every possible next token. Temperature reshapes these
probabilities before one is chosen: at temperature 0 the top row always wins.

> **Where we are.** You've loaded a model with Hugging Face, counted its parameters, applied its chat template, generated
> an answer, and looked at its probabilities for the next token. Finally, where models like this come from.

## 7. Where do these models come from? The foundation-model lifecycle

A **foundation model** is a large model trained on broad data that many different applications are then built on. You
won't train one; almost nobody does. But when you choose one, it helps to know what went into it.

| Stage | What happens | Scale for a 7-billion-parameter model |
|---|---|---|
| Data | Web text, books, code, filtered and de-duplicated | Trillions of tokens |
| Pre-training | Predict the next token, over and over | Thousands of GPUs for weeks |
| Instruction tuning | Train on examples of instructions and good answers | Tens of thousands of examples |
| Preference tuning | Train towards answers people rated as better | Human ratings, safety testing |
| Release | Weights, licence, model card | You download it here |

A back-of-envelope calculation shows why. Training takes roughly **6 × parameters × training tokens** floating-point
operations (FLOPs: single additions or multiplications of decimal numbers). Divide by how many operations a GPU can do
per second to get the time, and multiply by a rental price to get the cost:

In [30]:
params = 7e9                     # a 7-billion-parameter model...
tokens = 2e12                    # ...trained on 2 trillion tokens
flops = 6 * params * tokens      # floating-point operations needed

# One modern data-centre GPU does about 1e15 operations a second; realistically you get about 40% of that.
gpu_flops = 1e15 * 0.4
gpu_hours = flops / gpu_flops / 3600          # seconds -> hours
print(f"~{gpu_hours:,.0f} GPU-hours")

# At two typical cloud rental prices per GPU per hour.
for price in [2.0, 4.0]:
    cost_millions = gpu_hours * price / 1e6
    print(f"at US${price:.0f}/GPU-hour: about US${cost_millions:.2f} million, for one successful run, "
          f"before data, staff, failed runs and evaluation")

~58,333 GPU-hours
at US$2/GPU-hour: about US$0.12 million, for one successful run, before data, staff, failed runs and evaluation
at US$4/GPU-hour: about US$0.23 million, for one successful run, before data, staff, failed runs and evaluation


Real projects cost several times that once you add failed experiments, data work and people. That's why the Fund's
realistic options are to **use** open models (today), **adapt** them to its own documents (Day 8 fine-tuning, Day 9
retrieval), and **govern** them (Day 10), not to build one from scratch.

**Your turn (2).** Change the calculation for a 70-billion-parameter model trained on 15 trillion tokens, about the scale of
the largest open models. How many GPU-hours? How long would that take on 1,000 GPUs?

In [31]:
# Your code here
params = 7e9                     # a 7-billion-parameter model...
tokens = 15e12                    # ...trained on 2 trillion tokens
flops = 6 * params * tokens      # floating-point operations needed

# One modern data-centre GPU does about 1e15 operations a second; realistically you get about 40% of that.
gpu_flops = 1e15 * 0.4
gpu_hours = flops / gpu_flops / 3600          # seconds -> hours
print(f"~{gpu_hours:,.0f} GPU-hours")

# At two typical cloud rental prices per GPU per hour.
for price in [2.0, 4.0]:
    cost_millions = gpu_hours * price / 1e6
    print(f"at US${price:.0f}/GPU-hour: about US${cost_millions:.2f} million, for one successful run, "
          f"before data, staff, failed runs and evaluation")

~437,500 GPU-hours
at US$2/GPU-hour: about US$0.88 million, for one successful run, before data, staff, failed runs and evaluation
at US$4/GPU-hour: about US$1.75 million, for one successful run, before data, staff, failed runs and evaluation


## Wrap-up

You've run open models two ways, measured speed and tokens, seen temperature at work, looked inside a transformer, and seen
a model answer policy questions confidently and wrongly. In Lab 2 you'll learn to write prompts that make these models
useful and safe for Fund tasks.

---

## Looking back, looking ahead

**What you did in this lab.** You ran open models locally through Ollama, called them over HTTP, measured tokens and speed,
found that Swahili costs more tokens, saw temperature at work, watched models answer policy questions confidently and
wrongly, looked inside a small model with Hugging Face, and worked out why the Fund will use and adapt models rather than
train one.

**The ideas to carry forward.**

- An LLM predicts the next token, in a loop; everything else is built on that.
- Local models keep member data under the Fund's control; their size is limited by memory.
- Use temperature 0 for Fund tasks, and check `done_reason`.
- Never treat an LLM as the source of truth for policy.

**What happens next.** Lab 2: writing prompts that make these models follow instructions, answer from documents, return
structured output, and resist instructions hidden in the text they read.

## Suggested answers

**(1)** The same request with two length limits. `done_reason` tells you why the model stopped: `stop` means it finished
naturally, `length` means it hit the limit.

In [39]:
for limit in [500]:
    reply = chat("List all the benefits a social security fund might pay.", max_tokens=limit)
    # print("max_tokens =", limit, "| done_reason =", reply.get("done_reason"))
    print(reply["message"]["content"])        # the first 500 characters
    print()

A social security fund typically provides several types of benefits to individuals and families. Here is a list of some common benefits:

1. **Old Age Pension**: This benefit is designed for retired individuals who have reached the age of retirement as stipulated by the government.

2. **Unemployment Benefits**: These are paid out when an individual becomes unemployed due to reasons beyond their control, such as loss of employment or business closure.

3. **Healthcare Coverage**: Many social security funds provide coverage for medical expenses and healthcare services, which can help individuals manage health-related costs effectively.

4. **Disability Insurance**: This benefit is designed for individuals who become disabled and are unable to work due to illness or injury.

5. **Child Support Payments**: In some countries, a social security fund may also cover payments for child support if the parents have separated or divorced.

6. **Education Assistance**: Some funds provide assistanc

At 30 the answer stops mid-sentence and `done_reason` is `length`, not `stop`. If your application expects JSON, a cut-off
answer is broken JSON. Always check `done_reason` and set the limit with room to spare.

**(2)** The same formula with the bigger numbers. Dividing by 1,000 GPUs and 24 hours gives days:

In [40]:
flops_70b = 6 * 70e9 * 15e12
gpu_hours_70b = flops_70b / gpu_flops / 3600
days_on_1000_gpus = gpu_hours_70b / 1000 / 24

print(f"~{gpu_hours_70b / 1e6:.1f} million GPU-hours")
print(f"on 1,000 GPUs: about {days_on_1000_gpus:.0f} days")

~4.4 million GPU-hours
on 1,000 GPUs: about 182 days
